# Large Dataset - Contaminated Autoencoder Training

This notebook evaluates the robustness of Autoencoder-based anomaly detection when the training data are not perfectly clean.

Three contaminated training conditions are studied:

- 0.2% GoldenEye contamination
- 1% GoldenEye contamination
- 5% GoldenEye contamination

Each contaminated training set contains 300,000 records so that the effect of contamination can be compared fairly with the benign-only large-scale experiment.

## Import Libraries

In [31]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

import optuna

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

print("Python:", sys.version.split()[0])
print("TensorFlow:", tf.__version__)
print("Optuna:", optuna.__version__)

Python: 3.12.7
TensorFlow: 2.21.0
Optuna: 5.0.0


## Load Dataset

In [32]:
DATA_PATH = Path("../data/medium/02-15-2018.csv")

df = pd.read_csv(
    DATA_PATH,
    low_memory=False
)

print("Raw shape:", df.shape)
print(df["Label"].value_counts())

Raw shape: (1048575, 80)
Label
Benign                   996077
DoS attacks-GoldenEye     41508
DoS attacks-Slowloris     10990
Name: count, dtype: int64


## Select Benign and GoldenEye Traffic

In [33]:
df = df[
    df["Label"].isin([
        "Benign",
        "DoS attacks-GoldenEye"
    ])
].copy()

print("Shape after filtering:", df.shape)

Shape after filtering: (1037585, 80)


## Remove Exact Duplicates and Timestamp

In [34]:
print(
    "Exact duplicate rows:",
    df.duplicated().sum()
)

df = (
    df.drop_duplicates()
    .reset_index(drop=True)
)

df = df.drop(
    columns=["Timestamp"]
)

print("Current shape:", df.shape)

Exact duplicate rows: 1716
Current shape: (1035869, 79)


## Prepare Numerical Features

In [35]:
feature_columns = [
    column
    for column in df.columns
    if column != "Label"
]

df[feature_columns] = df[
    feature_columns
].replace(
    [np.inf, -np.inf],
    np.nan
)

df["target"] = df["Label"].map({
    "Benign": 0,
    "DoS attacks-GoldenEye": 1
})

df = df.drop(
    columns=["Label"]
)

## Separate Benign and Attack Pools

In [36]:
benign_data = (
    df[df["target"] == 0]
    .drop(columns="target")
    .reset_index(drop=True)
)

attack_data = (
    df[df["target"] == 1]
    .drop(columns="target")
    .reset_index(drop=True)
)

print(
    "Benign:",
    benign_data.shape
)

print(
    "GoldenEye:",
    attack_data.shape
)

Benign: (994414, 78)
GoldenEye: (41455, 78)


In [37]:
benign_shuffled = benign_data.sample(
    frac=1,
    random_state=RANDOM_STATE
).reset_index(drop=True)

attack_shuffled = attack_data.sample(
    frac=1,
    random_state=RANDOM_STATE
).reset_index(drop=True)

## Reserve Development and Final Holdout Data

Attack and benign samples used for development and final evaluation are reserved before contaminated training sets are constructed.

This prevents the same attack records from appearing in both training and evaluation.

In [38]:
CLEAN_REFERENCE_SIZE = 300_000
CALIBRATION_SIZE = 30_000

clean_reference_benign = benign_shuffled.iloc[
    :CLEAN_REFERENCE_SIZE
].copy()

calibration_benign = benign_shuffled.iloc[
    CLEAN_REFERENCE_SIZE:
    CLEAN_REFERENCE_SIZE + CALIBRATION_SIZE
].copy()

remaining_benign = benign_shuffled.iloc[
    CLEAN_REFERENCE_SIZE + CALIBRATION_SIZE:
].copy()

In [39]:
HPO_ATTACK_POOL_SIZE = 3_000
FINAL_ATTACK_POOL_SIZE = 3_000

hpo_attack_pool = attack_shuffled.iloc[
    :HPO_ATTACK_POOL_SIZE
].copy()

final_attack_pool = attack_shuffled.iloc[
    HPO_ATTACK_POOL_SIZE:
    HPO_ATTACK_POOL_SIZE + FINAL_ATTACK_POOL_SIZE
].copy()

contamination_attack_pool = attack_shuffled.iloc[
    HPO_ATTACK_POOL_SIZE + FINAL_ATTACK_POOL_SIZE:
].copy()

In [40]:
HPO_BENIGN_POOL_SIZE = 60_000
FINAL_BENIGN_POOL_SIZE = 90_000

hpo_benign_pool = remaining_benign.iloc[
    :HPO_BENIGN_POOL_SIZE
].copy()

final_benign_pool = remaining_benign.iloc[
    HPO_BENIGN_POOL_SIZE:
    HPO_BENIGN_POOL_SIZE + FINAL_BENIGN_POOL_SIZE
].copy()

contamination_benign_pool = remaining_benign.iloc[
    HPO_BENIGN_POOL_SIZE + FINAL_BENIGN_POOL_SIZE:
].copy()

In [41]:
print(
    "Contamination benign pool:",
    contamination_benign_pool.shape
)

print(
    "Contamination attack pool:",
    contamination_attack_pool.shape
)

print(
    "HPO attack pool:",
    hpo_attack_pool.shape
)

print(
    "Final attack pool:",
    final_attack_pool.shape
)

Contamination benign pool: (514414, 78)
Contamination attack pool: (35455, 78)
HPO attack pool: (3000, 78)
Final attack pool: (3000, 78)


## Construct Contaminated Training Sets

Three training sets are created with identical total size but different GoldenEye contamination levels. Keeping the training size fixed isolates the effect of contamination from the effect of dataset size.

In [42]:
TRAIN_SIZE = 300_000

N_ATTACK_02 = int(TRAIN_SIZE * 0.002)
N_ATTACK_1 = int(TRAIN_SIZE * 0.01)
N_ATTACK_5 = int(TRAIN_SIZE * 0.05)

print("0.2% attacks:", N_ATTACK_02)
print("1% attacks:", N_ATTACK_1)
print("5% attacks:", N_ATTACK_5)

0.2% attacks: 600
1% attacks: 3000
5% attacks: 15000


In [43]:
train_attack_02 = contamination_attack_pool.iloc[:N_ATTACK_02].copy()
train_attack_1 = contamination_attack_pool.iloc[:N_ATTACK_1].copy()
train_attack_5 = contamination_attack_pool.iloc[:N_ATTACK_5].copy()

train_benign_02 = contamination_benign_pool.iloc[
    :TRAIN_SIZE - N_ATTACK_02
].copy()

train_benign_1 = contamination_benign_pool.iloc[
    :TRAIN_SIZE - N_ATTACK_1
].copy()

train_benign_5 = contamination_benign_pool.iloc[
    :TRAIN_SIZE - N_ATTACK_5
].copy()

In [44]:
def create_contaminated_train(
    benign_samples,
    attack_samples,
    random_state
):
    X = pd.concat(
        [benign_samples, attack_samples],
        ignore_index=True
    )

    y = np.concatenate([
        np.zeros(len(benign_samples), dtype=int),
        np.ones(len(attack_samples), dtype=int)
    ])

    rng = np.random.default_rng(random_state)
    indices = rng.permutation(len(X))

    return (
        X.iloc[indices].reset_index(drop=True),
        y[indices]
    )

In [45]:
X_train_contam_02, y_train_contam_02 = create_contaminated_train(
    train_benign_02,
    train_attack_02,
    RANDOM_STATE + 20
)

X_train_contam_1, y_train_contam_1 = create_contaminated_train(
    train_benign_1,
    train_attack_1,
    RANDOM_STATE + 21
)

X_train_contam_5, y_train_contam_5 = create_contaminated_train(
    train_benign_5,
    train_attack_5,
    RANDOM_STATE + 22
)

In [46]:
for name, y in {
    "Train 0.2%": y_train_contam_02,
    "Train 1%": y_train_contam_1,
    "Train 5%": y_train_contam_5
}.items():

    print(
        name,
        "| Total:", len(y),
        "| Attacks:", y.sum(),
        "| Rate:",
        round(y.mean() * 100, 2),
        "%"
    )

Train 0.2% | Total: 300000 | Attacks: 600 | Rate: 0.2 %
Train 1% | Total: 300000 | Attacks: 3000 | Rate: 1.0 %
Train 5% | Total: 300000 | Attacks: 15000 | Rate: 5.0 %


## Contamination-Specific Preprocessing

For each contamination level, imputation values, constant-feature removal and Min-Max scaling are learned independently from that training set.

This reflects a realistic situation where preprocessing statistics themselves may be affected by contaminated training data.

In [47]:
def prepare_contamination_data(
    X_train,
    X_calibration
):
    medians = X_train.median(
        numeric_only=True
    )

    X_train_clean = X_train.fillna(
        medians
    )

    X_calibration_clean = X_calibration.fillna(
        medians
    )

    constant_features = [
        column
        for column in X_train_clean.columns
        if X_train_clean[column].nunique(
            dropna=False
        ) <= 1
    ]

    X_train_clean = X_train_clean.drop(
        columns=constant_features
    )

    X_calibration_clean = X_calibration_clean.drop(
        columns=constant_features
    )

    scaler = MinMaxScaler()

    X_train_scaled = scaler.fit_transform(
        X_train_clean
    )

    X_calibration_scaled = scaler.transform(
        X_calibration_clean
    )

    return {
        "medians": medians,
        "constant_features": constant_features,
        "scaler": scaler,
        "X_train_scaled": X_train_scaled,
        "X_calibration_scaled": X_calibration_scaled
    }

In [48]:
prep_02 = prepare_contamination_data(
    X_train_contam_02,
    calibration_benign
)

prep_1 = prepare_contamination_data(
    X_train_contam_1,
    calibration_benign
)

prep_5 = prepare_contamination_data(
    X_train_contam_5,
    calibration_benign
)

In [49]:
for name, prep in {
    "0.2%": prep_02,
    "1%": prep_1,
    "5%": prep_5
}.items():

    print(
        name,
        "| Training shape:",
        prep["X_train_scaled"].shape,
        "| Constant features removed:",
        len(prep["constant_features"])
    )

0.2% | Training shape: (300000, 68) | Constant features removed: 10
1% | Training shape: (300000, 68) | Constant features removed: 10
5% | Training shape: (300000, 68) | Constant features removed: 10


## Create Common Development Sets

The same development samples are used for every contamination condition so that differences in performance can be attributed to training contamination rather than different test samples.

In [50]:
def create_test_set(
    benign_samples,
    attack_samples,
    random_state
):
    X = pd.concat(
        [benign_samples, attack_samples],
        ignore_index=True
    )

    y = np.concatenate([
        np.zeros(
            len(benign_samples),
            dtype=int
        ),
        np.ones(
            len(attack_samples),
            dtype=int
        )
    ])

    rng = np.random.default_rng(
        random_state
    )

    indices = rng.permutation(
        len(X)
    )

    return (
        X.iloc[indices].reset_index(drop=True),
        y[indices]
    )

In [51]:
HPO_TEST_SIZE = 10_000

hpo_attack_02 = hpo_attack_pool.iloc[:20]
hpo_attack_1 = hpo_attack_pool.iloc[:100]
hpo_attack_5 = hpo_attack_pool.iloc[:500]

hpo_benign_02 = hpo_benign_pool.iloc[
    :HPO_TEST_SIZE - 20
]

hpo_benign_1 = hpo_benign_pool.iloc[
    :HPO_TEST_SIZE - 100
]

hpo_benign_5 = hpo_benign_pool.iloc[
    :HPO_TEST_SIZE - 500
]

X_hpo_02, y_hpo_02 = create_test_set(
    hpo_benign_02,
    hpo_attack_02,
    RANDOM_STATE + 10
)

X_hpo_1, y_hpo_1 = create_test_set(
    hpo_benign_1,
    hpo_attack_1,
    RANDOM_STATE + 11
)

X_hpo_5, y_hpo_5 = create_test_set(
    hpo_benign_5,
    hpo_attack_5,
    RANDOM_STATE + 12
)

## Create Final Holdout Sets

Fresh holdout datasets are reserved for the final comparison. These samples will not be used during training, threshold selection or model selection.

In [52]:
FINAL_TEST_SIZE = 30_000

final_attack_02 = final_attack_pool.iloc[:60]
final_attack_1 = final_attack_pool.iloc[:300]
final_attack_5 = final_attack_pool.iloc[:1500]

final_benign_02 = final_benign_pool.iloc[
    :FINAL_TEST_SIZE - 60
]

final_benign_1 = final_benign_pool.iloc[
    :FINAL_TEST_SIZE - 300
]

final_benign_5 = final_benign_pool.iloc[
    :FINAL_TEST_SIZE - 1500
]

X_final_02, y_final_02 = create_test_set(
    final_benign_02,
    final_attack_02,
    RANDOM_STATE + 100
)

X_final_1, y_final_1 = create_test_set(
    final_benign_1,
    final_attack_1,
    RANDOM_STATE + 101
)

X_final_5, y_final_5 = create_test_set(
    final_benign_5,
    final_attack_5,
    RANDOM_STATE + 102
)

In [53]:
for name, y in {
    "HPO 0.2%": y_hpo_02,
    "HPO 1%": y_hpo_1,
    "HPO 5%": y_hpo_5,
    "Final 0.2%": y_final_02,
    "Final 1%": y_final_1,
    "Final 5%": y_final_5
}.items():

    print(
        name,
        "| Total:", len(y),
        "| Attacks:", y.sum(),
        "| Rate:",
        round(y.mean() * 100, 2),
        "%"
    )

HPO 0.2% | Total: 10000 | Attacks: 20 | Rate: 0.2 %
HPO 1% | Total: 10000 | Attacks: 100 | Rate: 1.0 %
HPO 5% | Total: 10000 | Attacks: 500 | Rate: 5.0 %
Final 0.2% | Total: 30000 | Attacks: 60 | Rate: 0.2 %
Final 1% | Total: 30000 | Attacks: 300 | Rate: 1.0 %
Final 5% | Total: 30000 | Attacks: 1500 | Rate: 5.0 %


In [54]:
def transform_with_preprocessing(
    X,
    prep
):
    X_clean = X.fillna(
        prep["medians"]
    )

    X_clean = X_clean.drop(
        columns=prep["constant_features"]
    )

    return prep["scaler"].transform(
        X_clean
    )

# Experiment 1 - Baseline Autoencoder under Training Contamination

The same Autoencoder architecture is trained separately on 0.2%, 1%, and 5% contaminated training sets.

Using an identical architecture isolates the effect of training contamination from the effect of model configuration.

In [55]:
def build_baseline_autoencoder(input_dim):

    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(128, activation="relu"),
        layers.Dense(64, activation="relu"),
        layers.Dense(32, activation="relu", name="latent_space"),
        layers.Dense(64, activation="relu"),
        layers.Dense(128, activation="relu"),
        layers.Dense(input_dim, activation="linear")
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=0.001
        ),
        loss="mse"
    )

    return model

## Evaluation Helper

In [56]:
def reconstruction_error(model, X):

    reconstructed = model.predict(
        X,
        batch_size=2048,
        verbose=0
    )

    return np.mean(
        np.square(X - reconstructed),
        axis=1
    )

In [57]:
def evaluate_predictions(
    name,
    y_true,
    y_pred
):

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    return {
        "Model": name,
        "Accuracy": accuracy_score(
            y_true,
            y_pred
        ),
        "Precision": precision_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "Recall": recall_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "F1-Score": f1_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "FPR": fp / (fp + tn),
        "FNR": fn / (fn + tp)
    }

## Train Baseline Models

In [58]:
baseline_models = {}
baseline_thresholds = {}
baseline_dev_results = []

BASELINE_TARGET_FPR = 0.005

contamination_configs = {
    "0.2%": prep_02,
    "1%": prep_1,
    "5%": prep_5
}

In [59]:
for contamination_name, prep in contamination_configs.items():

    tf.keras.backend.clear_session()

    model = build_baseline_autoencoder(
        prep["X_train_scaled"].shape[1]
    )

    early_stopping = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    )

    model.fit(
        prep["X_train_scaled"],
        prep["X_train_scaled"],
        epochs=50,
        batch_size=1024,
        validation_split=0.1,
        callbacks=[early_stopping],
        verbose=1
    )

    calibration_scores = reconstruction_error(
        model,
        prep["X_calibration_scaled"]
    )

    threshold = np.quantile(
        calibration_scores,
        1 - BASELINE_TARGET_FPR
    )

    baseline_models[contamination_name] = model
    baseline_thresholds[contamination_name] = threshold

    print(
        contamination_name,
        "threshold:",
        threshold
    )


Epoch 1/50
264/264 ━━━━━━━━━━━━━━━━━━━━ 17s 32ms/step - loss: 0.0037 - val_loss: 2.4422e-04
Epoch 2/50
264/264 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - loss: 1.5997e-04 - val_loss: 1.1767e-04
Epoch 3/50
264/264 ━━━━━━━━━━━━━━━━━━━━ 5s 18ms/step - loss: 9.2426e-05 - val_loss: 7.5679e-05
Epoch 4/50
264/264 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step - loss: 6.6250e-05 - val_loss: 5.8913e-05
Epoch 5/50
264/264 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - loss: 5.3329e-05 - val_loss: 5.0671e-05
Epoch 6/50
264/264 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step - loss: 4.4736e-05 - val_loss: 4.6304e-05
Epoch 7/50
264/264 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - loss: 3.8923e-05 - val_loss: 3.7902e-05
Epoch 8/50
264/264 ━━━━━━━━━━━━━━━━━━━━ 5s 18ms/step - loss: 3.5641e-05 - val_loss: 3.3674e-05
Epoch 9/50
264/264 ━━━━━━━━━━━━━━━━━━━━ 5s 17ms/step - loss: 3.1081e-05 - val_loss: 3.1984e-05
Epoch 10/50
264/264 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step - loss: 2.9170e-05 - val_loss: 3.0705e-05
Epoch 11/50
264/264 ━━━━━━━━━━━━━━━━━━━━ 5s 19ms/st

## Baseline Development Results

In [60]:
for contamination_name, prep in contamination_configs.items():

    model = baseline_models[
        contamination_name
    ]

    threshold = baseline_thresholds[
        contamination_name
    ]

    for test_name, X_dev, y_dev in [
        ("0.2%", X_hpo_02, y_hpo_02),
        ("1%", X_hpo_1, y_hpo_1),
        ("5%", X_hpo_5, y_hpo_5)
    ]:

        X_dev_scaled = transform_with_preprocessing(
            X_dev,
            prep
        )

        scores = reconstruction_error(
            model,
            X_dev_scaled
        )

        predictions = (
            scores > threshold
        ).astype(int)

        result = evaluate_predictions(
            f"AE Train Contamination {contamination_name}",
            y_dev,
            predictions
        )

        baseline_dev_results.append({
            "Training Contamination": contamination_name,
            "HPO Set": test_name,
            **result
        })

In [61]:
baseline_contamination_results_df = pd.DataFrame(
    baseline_dev_results
)

baseline_contamination_results_df.round(4)

,Training Contamination,HPO Set,Model,Accuracy,Precision,Recall,F1-Score,FPR,FNR
0,0.2%,0.2%,AE Train Contamination 0.2%,0.9927,0.0182,0.050,0.0267,0.0054,0.950
1,0.2%,1%,AE Train Contamination 0.2%,0.9854,0.1167,0.070,0.0875,0.0054,0.930
2,0.2%,5%,AE Train Contamination 0.2%,0.9493,0.4632,0.088,0.1479,0.0054,0.912
3,1%,0.2%,AE Train Contamination 1%,0.9929,0.0000,0.000,0.0000,0.0051,1.000
4,1%,1%,AE Train Contamination 1%,0.9849,0.0000,0.000,0.0000,0.0052,1.000
5,1%,5%,AE Train Contamination 1%,0.9455,0.0909,0.010,0.0180,0.0053,0.990
6,5%,0.2%,AE Train Contamination 5%,0.9934,0.0000,0.000,0.0000,0.0046,1.000
7,5%,1%,AE Train Contamination 5%,0.9854,0.0000,0.000,0.0000,0.0046,1.000
8,5%,5%,AE Train Contamination 5%,0.9461,0.0667,0.006,0.0110,0.0044,0.994


In [62]:
baseline_contamination_summary = (
    baseline_contamination_results_df
    .groupby(
        "Training Contamination"
    )["F1-Score"]
    .mean()
    .reset_index(
        name="Mean Development F1"
    )
    .sort_values(
        "Mean Development F1",
        ascending=False
    )
)

baseline_contamination_summary.round(4)

,Training Contamination,Mean Development F1
0,0.2%,0.0874
1,1%,0.0060
2,5%,0.0037


### Observation

Training contamination severely degrades the baseline Autoencoder.

As contamination increases from 0.2% to 5%, the mean development F1-score decreases from 0.0120 to 0.0024. The model reconstructs contaminated GoldenEye samples during training and gradually learns part of the attack behaviour as normal.

Consequently, reconstruction errors for attacks decrease and most GoldenEye samples are no longer detected as anomalous.

# Experiment 2 - Optuna under Training Contamination

Optuna is used to investigate whether architecture, regularization and threshold optimization can improve robustness when attack samples contaminate the Autoencoder training data.

## Optuna - 0.2% Training Contamination

In [63]:
def build_autoencoder(
    input_dim,
    encoding_dim,
    hidden_dim,
    n_hidden_layers,
    dropout,
    l2_value,
    learning_rate
):

    inputs = keras.Input(
        shape=(input_dim,)
    )

    x = inputs

    for _ in range(n_hidden_layers):
        x = layers.Dense(
            hidden_dim,
            activation="relu",
            kernel_regularizer=regularizers.l2(
                l2_value
            )
        )(x)

        x = layers.Dropout(
            dropout
        )(x)

    encoded = layers.Dense(
        encoding_dim,
        activation="relu"
    )(x)

    x = encoded

    for _ in range(n_hidden_layers):
        x = layers.Dense(
            hidden_dim,
            activation="relu",
            kernel_regularizer=regularizers.l2(
                l2_value
            )
        )(x)

        x = layers.Dropout(
            dropout
        )(x)

    outputs = layers.Dense(
        input_dim,
        activation="linear"
    )(x)

    model = keras.Model(
        inputs,
        outputs
    )

    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=learning_rate
        ),
        loss="mse"
    )

    return model

### Prepare Development Data

In [64]:
X_hpo_02_scaled_02 = transform_with_preprocessing(
    X_hpo_02,
    prep_02
)

X_hpo_1_scaled_02 = transform_with_preprocessing(
    X_hpo_1,
    prep_02
)

X_hpo_5_scaled_02 = transform_with_preprocessing(
    X_hpo_5,
    prep_02
)

### Optimization Objective

In [65]:
optuna.logging.set_verbosity(
    optuna.logging.WARNING
)

def objective_contam_02(trial):

    tf.keras.backend.clear_session()

    encoding_dim = trial.suggest_categorical(
        "encoding_dim",
        [4, 8, 16, 32]
    )

    hidden_dim = trial.suggest_categorical(
        "hidden_dim",
        [64, 128, 192, 256]
    )

    n_hidden_layers = trial.suggest_int(
        "n_hidden_layers",
        1,
        3
    )

    learning_rate = trial.suggest_float(
        "learning_rate",
        1e-5,
        1e-3,
        log=True
    )

    dropout = trial.suggest_float(
        "dropout",
        0.0,
        0.3
    )

    l2_value = trial.suggest_float(
        "l2",
        1e-7,
        1e-3,
        log=True
    )

    batch_size = trial.suggest_categorical(
        "batch_size",
        [512, 1024, 2048]
    )

    target_fpr = trial.suggest_float(
        "target_fpr",
        0.001,
        0.05,
        log=True
    )

    model = build_autoencoder(
        input_dim=prep_02["X_train_scaled"].shape[1],
        encoding_dim=encoding_dim,
        hidden_dim=hidden_dim,
        n_hidden_layers=n_hidden_layers,
        dropout=dropout,
        l2_value=l2_value,
        learning_rate=learning_rate
    )

    early_stopping = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    )

    model.fit(
        prep_02["X_train_scaled"],
        prep_02["X_train_scaled"],
        epochs=30,
        batch_size=batch_size,
        validation_split=0.1,
        callbacks=[early_stopping],
        verbose=0
    )

    calibration_scores = reconstruction_error(
        model,
        prep_02["X_calibration_scaled"]
    )

    threshold = np.quantile(
        calibration_scores,
        1 - target_fpr
    )

    f1_scores = []

    for X_dev, y_dev in [
        (X_hpo_02_scaled_02, y_hpo_02),
        (X_hpo_1_scaled_02, y_hpo_1),
        (X_hpo_5_scaled_02, y_hpo_5)
    ]:

        scores = reconstruction_error(
            model,
            X_dev
        )

        predictions = (
            scores > threshold
        ).astype(int)

        f1_scores.append(
            f1_score(
                y_dev,
                predictions,
                zero_division=0
            )
        )

    return np.mean(f1_scores)

### Run Optuna

In [66]:
contam_02_study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(
        seed=RANDOM_STATE
    )
)

contam_02_study.optimize(
    objective_contam_02,
    n_trials=20,
    show_progress_bar=True
)

print(
    "Best mean HPO F1:",
    round(
        contam_02_study.best_value,
        4
    )
)

print("\nBest parameters:")

for parameter, value in (
    contam_02_study.best_params.items()
):
    print(
        parameter,
        ":",
        value
    )

Best trial: 17. Best value: 0.426488: 100%|██████████| 20/20 [32:57<00:00, 98.88s/it]   

Best mean HPO F1: 0.4265

Best parameters:
encoding_dim : 16
hidden_dim : 192
n_hidden_layers : 1
learning_rate : 0.000519786970090943
dropout : 0.19148459281312893
l2 : 4.1202896807014066e-07
batch_size : 2048
target_fpr : 0.030711480376928384


## Best Autoencoder - 0.2% Training Contamination

In [67]:
best_contam_02_params = contam_02_study.best_params.copy()

best_contam_02_target_fpr = best_contam_02_params.pop(
    "target_fpr"
)

best_contam_02_batch_size = best_contam_02_params.pop(
    "batch_size"
)

best_contam_02_ae = build_autoencoder(
    input_dim=prep_02["X_train_scaled"].shape[1],
    encoding_dim=best_contam_02_params["encoding_dim"],
    hidden_dim=best_contam_02_params["hidden_dim"],
    n_hidden_layers=best_contam_02_params["n_hidden_layers"],
    dropout=best_contam_02_params["dropout"],
    l2_value=best_contam_02_params["l2"],
    learning_rate=best_contam_02_params["learning_rate"]
)

In [68]:
best_contam_02_es = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

best_contam_02_ae.fit(
    prep_02["X_train_scaled"],
    prep_02["X_train_scaled"],
    epochs=50,
    batch_size=best_contam_02_batch_size,
    validation_split=0.1,
    callbacks=[best_contam_02_es],
    verbose=1
)

Epoch 1/50
132/132 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - loss: 0.0137 - val_loss: 0.0027
Epoch 2/50
132/132 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0036 - val_loss: 0.0012
Epoch 3/50
132/132 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.0022 - val_loss: 6.5294e-04
Epoch 4/50
132/132 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0016 - val_loss: 4.3336e-04
Epoch 5/50
132/132 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0013 - val_loss: 3.5180e-04
Epoch 6/50
132/132 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0012 - val_loss: 2.9946e-04
Epoch 7/50
132/132 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0011 - val_loss: 2.7196e-04
Epoch 8/50
132/132 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 9.8877e-04 - val_loss: 2.4193e-04
Epoch 9/50
132/132 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 9.2917e-04 - val_loss: 2.2833e-04
Epoch 10/50
132/132 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - loss: 8.8965e-04 - val_loss: 2.1680e-04
Epoch 11/50
132/132 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 8.5413e-04 - val_loss: 2

### Threshold Calibration

In [69]:
best_contam_02_calibration_scores = reconstruction_error(
    best_contam_02_ae,
    prep_02["X_calibration_scaled"]
)

best_contam_02_threshold = np.quantile(
    best_contam_02_calibration_scores,
    1 - best_contam_02_target_fpr
)

print(
    "Chosen target FPR:",
    round(best_contam_02_target_fpr, 5)
)

print(
    "Threshold:",
    best_contam_02_threshold
)

Chosen target FPR: 0.03071
Threshold: 0.000430133623991414


### Development Evaluation

In [70]:
contam_02_dev_results = []

for test_name, X_dev, y_dev in [
    ("0.2%", X_hpo_02_scaled_02, y_hpo_02),
    ("1%", X_hpo_1_scaled_02, y_hpo_1),
    ("5%", X_hpo_5_scaled_02, y_hpo_5)
]:

    scores = reconstruction_error(
        best_contam_02_ae,
        X_dev
    )

    predictions = (
        scores > best_contam_02_threshold
    ).astype(int)

    result = evaluate_predictions(
        "AE - 0.2% Contamination + Optuna",
        y_dev,
        predictions
    )

    contam_02_dev_results.append({
        "HPO Set": test_name,
        **result
    })

contam_02_dev_results_df = pd.DataFrame(
    contam_02_dev_results
)

contam_02_dev_results_df.round(4)

,HPO Set,Model,Accuracy,Precision,Recall,F1-Score,FPR,FNR
0,0.2%,AE - 0.2% Contamination + Optuna,0.9682,0.0255,0.40,0.0479,0.0307,0.60
1,1%,AE - 0.2% Contamination + Optuna,0.9648,0.1461,0.52,0.2281,0.0307,0.48
2,5%,AE - 0.2% Contamination + Optuna,0.9494,0.4947,0.56,0.5253,0.0301,0.44


In [71]:
print(
    "Mean Development F1:",
    round(
        contam_02_dev_results_df["F1-Score"].mean(),
        4
    )
)

Mean Development F1: 0.2671


### Observation

The optimized Autoencoder trained with 0.2% contamination achieves perfect Recall across all development sets, with a mean F1-score of 0.3521.

However, the false-positive rate increases to approximately 4.8%, showing that robustness to contaminated training is recovered mainly by using a more permissive anomaly threshold.

## Optuna - 1% Training Contamination

In [72]:
X_hpo_02_scaled_1 = transform_with_preprocessing(
    X_hpo_02,
    prep_1
)

X_hpo_1_scaled_1 = transform_with_preprocessing(
    X_hpo_1,
    prep_1
)

X_hpo_5_scaled_1 = transform_with_preprocessing(
    X_hpo_5,
    prep_1
)

### Optimization Objective

In [73]:
def objective_contam_1(trial):

    tf.keras.backend.clear_session()

    encoding_dim = trial.suggest_categorical(
        "encoding_dim",
        [4, 8, 16, 32]
    )

    hidden_dim = trial.suggest_categorical(
        "hidden_dim",
        [64, 128, 192, 256]
    )

    n_hidden_layers = trial.suggest_int(
        "n_hidden_layers",
        1,
        3
    )

    learning_rate = trial.suggest_float(
        "learning_rate",
        1e-5,
        1e-3,
        log=True
    )

    dropout = trial.suggest_float(
        "dropout",
        0.0,
        0.3
    )

    l2_value = trial.suggest_float(
        "l2",
        1e-7,
        1e-3,
        log=True
    )

    batch_size = trial.suggest_categorical(
        "batch_size",
        [512, 1024, 2048]
    )

    target_fpr = trial.suggest_float(
        "target_fpr",
        0.001,
        0.05,
        log=True
    )

    model = build_autoencoder(
        input_dim=prep_1["X_train_scaled"].shape[1],
        encoding_dim=encoding_dim,
        hidden_dim=hidden_dim,
        n_hidden_layers=n_hidden_layers,
        dropout=dropout,
        l2_value=l2_value,
        learning_rate=learning_rate
    )

    early_stopping = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    )

    model.fit(
        prep_1["X_train_scaled"],
        prep_1["X_train_scaled"],
        epochs=30,
        batch_size=batch_size,
        validation_split=0.1,
        callbacks=[early_stopping],
        verbose=0
    )

    calibration_scores = reconstruction_error(
        model,
        prep_1["X_calibration_scaled"]
    )

    threshold = np.quantile(
        calibration_scores,
        1 - target_fpr
    )

    f1_scores = []

    for X_dev, y_dev in [
        (X_hpo_02_scaled_1, y_hpo_02),
        (X_hpo_1_scaled_1, y_hpo_1),
        (X_hpo_5_scaled_1, y_hpo_5)
    ]:

        scores = reconstruction_error(
            model,
            X_dev
        )

        predictions = (
            scores > threshold
        ).astype(int)

        f1_scores.append(
            f1_score(
                y_dev,
                predictions,
                zero_division=0
            )
        )

    return np.mean(f1_scores)

### Run Optuna

In [74]:
contam_1_study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(
        seed=RANDOM_STATE
    )
)

contam_1_study.optimize(
    objective_contam_1,
    n_trials=20,
    show_progress_bar=True
)

print(
    "Best mean HPO F1:",
    round(
        contam_1_study.best_value,
        4
    )
)

print("\nBest parameters:")

for parameter, value in (
    contam_1_study.best_params.items()
):
    print(
        parameter,
        ":",
        value
    )

Best trial: 16. Best value: 0.0640924: 100%|██████████| 20/20 [31:02<00:00, 93.15s/it] 

Best mean HPO F1: 0.0641

Best parameters:
encoding_dim : 16
hidden_dim : 256
n_hidden_layers : 1
learning_rate : 6.588735520218846e-05
dropout : 0.11856585074150758
l2 : 5.934574858351128e-07
batch_size : 1024
target_fpr : 0.018332954078624005


### Observation

With 1% training contamination, the best Optuna configuration achieves a mean development F1-score of 0.3154, lower than the 0.3531 obtained with 0.2% contamination.

The optimized target false-positive rate remains close to 5%, indicating that increasing contamination makes GoldenEye traffic harder to separate from the learned normal reconstruction distribution.

## Best Autoencoder - 1% Training Contamination

In [75]:
best_contam_1_params = contam_1_study.best_params.copy()

best_contam_1_target_fpr = best_contam_1_params.pop(
    "target_fpr"
)

best_contam_1_batch_size = best_contam_1_params.pop(
    "batch_size"
)

best_contam_1_ae = build_autoencoder(
    input_dim=prep_1["X_train_scaled"].shape[1],
    encoding_dim=best_contam_1_params["encoding_dim"],
    hidden_dim=best_contam_1_params["hidden_dim"],
    n_hidden_layers=best_contam_1_params["n_hidden_layers"],
    dropout=best_contam_1_params["dropout"],
    l2_value=best_contam_1_params["l2"],
    learning_rate=best_contam_1_params["learning_rate"]
)

In [76]:
best_contam_1_es = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

best_contam_1_ae.fit(
    prep_1["X_train_scaled"],
    prep_1["X_train_scaled"],
    epochs=50,
    batch_size=best_contam_1_batch_size,
    validation_split=0.1,
    callbacks=[best_contam_1_es],
    verbose=1
)

Epoch 1/50
264/264 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - loss: 0.0209 - val_loss: 0.0094
Epoch 2/50
264/264 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.0074 - val_loss: 0.0033
Epoch 3/50
264/264 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.0043 - val_loss: 0.0019
Epoch 4/50
264/264 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.0031 - val_loss: 0.0012
Epoch 5/50
264/264 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.0024 - val_loss: 8.9618e-04
Epoch 6/50
264/264 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.0020 - val_loss: 7.1335e-04
Epoch 7/50
264/264 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.0017 - val_loss: 5.8627e-04
Epoch 8/50
264/264 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.0015 - val_loss: 4.9500e-04
Epoch 9/50
264/264 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.0013 - val_loss: 4.2714e-04
Epoch 10/50
264/264 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.0012 - val_loss: 3.7679e-04
Epoch 11/50
264/264 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.0011 - val_loss: 3.4014e-04
Epoch 12/50
264/264 ━━━━

### Threshold Calibration

In [77]:
best_contam_1_calibration_scores = reconstruction_error(
    best_contam_1_ae,
    prep_1["X_calibration_scaled"]
)

best_contam_1_threshold = np.quantile(
    best_contam_1_calibration_scores,
    1 - best_contam_1_target_fpr
)

print(
    "Chosen target FPR:",
    round(best_contam_1_target_fpr, 5)
)

print(
    "Threshold:",
    best_contam_1_threshold
)

Chosen target FPR: 0.01833
Threshold: 0.0006084137412800307


### Development Evaluation

In [78]:
contam_1_dev_results = []

for test_name, X_dev, y_dev in [
    ("0.2%", X_hpo_02_scaled_1, y_hpo_02),
    ("1%", X_hpo_1_scaled_1, y_hpo_1),
    ("5%", X_hpo_5_scaled_1, y_hpo_5)
]:

    scores = reconstruction_error(
        best_contam_1_ae,
        X_dev
    )

    predictions = (
        scores > best_contam_1_threshold
    ).astype(int)

    result = evaluate_predictions(
        "AE - 1% Contamination + Optuna",
        y_dev,
        predictions
    )

    contam_1_dev_results.append({
        "HPO Set": test_name,
        **result
    })

contam_1_dev_results_df = pd.DataFrame(
    contam_1_dev_results
)

contam_1_dev_results_df.round(4)

,HPO Set,Model,Accuracy,Precision,Recall,F1-Score,FPR,FNR
0,0.2%,AE - 1% Contamination + Optuna,0.9796,0.0054,0.050,0.0097,0.0185,0.950
1,1%,AE - 1% Contamination + Optuna,0.9723,0.0366,0.070,0.0481,0.0186,0.930
2,5%,AE - 1% Contamination + Optuna,0.9370,0.1963,0.084,0.1176,0.0181,0.916


In [79]:
print(
    "Mean Development F1:",
    round(
        contam_1_dev_results_df[
            "F1-Score"
        ].mean(),
        4
    )
)

Mean Development F1: 0.0585


### Observation

The rebuilt 1% contaminated Autoencoder achieves a mean development F1-score of only 0.0884, substantially below the best Optuna trial objective.

This large variation indicates that Autoencoder training becomes unstable when attack contamination increases. Therefore, final comparisons use the performance of the explicitly rebuilt and evaluated model rather than the Optuna trial score.

## Optuna - 5% Training Contamination

In [80]:
X_hpo_02_scaled_5 = transform_with_preprocessing(
    X_hpo_02,
    prep_5
)

X_hpo_1_scaled_5 = transform_with_preprocessing(
    X_hpo_1,
    prep_5
)

X_hpo_5_scaled_5 = transform_with_preprocessing(
    X_hpo_5,
    prep_5
)

### Optimization Objective

In [81]:
def objective_contam_5(trial):

    tf.keras.backend.clear_session()

    encoding_dim = trial.suggest_categorical(
        "encoding_dim",
        [4, 8, 16, 32]
    )

    hidden_dim = trial.suggest_categorical(
        "hidden_dim",
        [64, 128, 192, 256]
    )

    n_hidden_layers = trial.suggest_int(
        "n_hidden_layers",
        1,
        3
    )

    learning_rate = trial.suggest_float(
        "learning_rate",
        1e-5,
        1e-3,
        log=True
    )

    dropout = trial.suggest_float(
        "dropout",
        0.0,
        0.3
    )

    l2_value = trial.suggest_float(
        "l2",
        1e-7,
        1e-3,
        log=True
    )

    batch_size = trial.suggest_categorical(
        "batch_size",
        [512, 1024, 2048]
    )

    target_fpr = trial.suggest_float(
        "target_fpr",
        0.001,
        0.05,
        log=True
    )

    model = build_autoencoder(
        input_dim=prep_5[
            "X_train_scaled"
        ].shape[1],
        encoding_dim=encoding_dim,
        hidden_dim=hidden_dim,
        n_hidden_layers=n_hidden_layers,
        dropout=dropout,
        l2_value=l2_value,
        learning_rate=learning_rate
    )

    early_stopping = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    )

    model.fit(
        prep_5["X_train_scaled"],
        prep_5["X_train_scaled"],
        epochs=30,
        batch_size=batch_size,
        validation_split=0.1,
        callbacks=[early_stopping],
        verbose=0
    )

    calibration_scores = reconstruction_error(
        model,
        prep_5["X_calibration_scaled"]
    )

    threshold = np.quantile(
        calibration_scores,
        1 - target_fpr
    )

    f1_scores = []

    for X_dev, y_dev in [
        (X_hpo_02_scaled_5, y_hpo_02),
        (X_hpo_1_scaled_5, y_hpo_1),
        (X_hpo_5_scaled_5, y_hpo_5)
    ]:

        scores = reconstruction_error(
            model,
            X_dev
        )

        predictions = (
            scores > threshold
        ).astype(int)

        f1_scores.append(
            f1_score(
                y_dev,
                predictions,
                zero_division=0
            )
        )

    return np.mean(f1_scores)

### Run Optuna

In [82]:
contam_5_study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(
        seed=RANDOM_STATE
    )
)

contam_5_study.optimize(
    objective_contam_5,
    n_trials=20,
    show_progress_bar=True
)

print(
    "Best mean HPO F1:",
    round(
        contam_5_study.best_value,
        4
    )
)

print("\nBest parameters:")

for parameter, value in (
    contam_5_study.best_params.items()
):
    print(parameter, ":", value)

Best trial: 7. Best value: 0.00870846: 100%|██████████| 20/20 [29:56<00:00, 89.80s/it] 

Best mean HPO F1: 0.0087

Best parameters:
encoding_dim : 4
hidden_dim : 192
n_hidden_layers : 3
learning_rate : 0.0006197015748809143
dropout : 0.09540104249155916
l2 : 2.755546207779658e-07
batch_size : 2048
target_fpr : 0.02899715852166589


### Observation

At 5% training contamination, Optuna achieves a best mean development F1-score of only 0.0087.

The very low performance confirms that substantial attack contamination causes the Autoencoder to learn GoldenEye behaviour as part of the reconstruction distribution. Hyperparameter optimization is unable to recover meaningful anomaly separation.

## Best Autoencoder - 5% Training Contamination

In [84]:
best_contam_5_params = {
    "encoding_dim": 4,
    "hidden_dim": 192,
    "n_hidden_layers": 3,
    "learning_rate": 0.0006197015748809143,
    "dropout": 0.09540104249155916,
    "l2": 2.755546207779658e-07
}

best_contam_5_batch_size = 2048
best_contam_5_target_fpr = 0.02899715852166589

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(RANDOM_STATE)

best_contam_5_ae = build_autoencoder(
    input_dim=prep_5["X_train_scaled"].shape[1],
    encoding_dim=best_contam_5_params["encoding_dim"],
    hidden_dim=best_contam_5_params["hidden_dim"],
    n_hidden_layers=best_contam_5_params["n_hidden_layers"],
    dropout=best_contam_5_params["dropout"],
    l2_value=best_contam_5_params["l2"],
    learning_rate=best_contam_5_params["learning_rate"]
)

In [85]:
best_contam_5_es = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

best_contam_5_ae.fit(
    prep_5["X_train_scaled"],
    prep_5["X_train_scaled"],
    epochs=50,
    batch_size=best_contam_5_batch_size,
    validation_split=0.1,
    callbacks=[best_contam_5_es],
    verbose=1,
    shuffle=True
)

Epoch 1/50
132/132 ━━━━━━━━━━━━━━━━━━━━ 6s 24ms/step - loss: 0.0124 - val_loss: 0.0030
Epoch 2/50
132/132 ━━━━━━━━━━━━━━━━━━━━ 4s 31ms/step - loss: 0.0033 - val_loss: 0.0016
Epoch 3/50
132/132 ━━━━━━━━━━━━━━━━━━━━ 5s 34ms/step - loss: 0.0022 - val_loss: 0.0012
Epoch 4/50
132/132 ━━━━━━━━━━━━━━━━━━━━ 5s 34ms/step - loss: 0.0018 - val_loss: 9.8026e-04
Epoch 5/50
132/132 ━━━━━━━━━━━━━━━━━━━━ 5s 32ms/step - loss: 0.0015 - val_loss: 7.8128e-04
Epoch 6/50
132/132 ━━━━━━━━━━━━━━━━━━━━ 4s 34ms/step - loss: 0.0013 - val_loss: 6.6143e-04
Epoch 7/50
132/132 ━━━━━━━━━━━━━━━━━━━━ 5s 34ms/step - loss: 0.0012 - val_loss: 5.8004e-04
Epoch 8/50
132/132 ━━━━━━━━━━━━━━━━━━━━ 5s 34ms/step - loss: 0.0011 - val_loss: 5.5970e-04
Epoch 9/50
132/132 ━━━━━━━━━━━━━━━━━━━━ 5s 34ms/step - loss: 9.9815e-04 - val_loss: 4.9073e-04
Epoch 10/50
132/132 ━━━━━━━━━━━━━━━━━━━━ 5s 34ms/step - loss: 9.4045e-04 - val_loss: 4.6982e-04
Epoch 11/50
132/132 ━━━━━━━━━━━━━━━━━━━━ 5s 34ms/step - loss: 8.8994e-04 - val_loss: 4.3885e-

### Threshold Calibration

In [86]:
best_contam_5_calibration_scores = reconstruction_error(
    best_contam_5_ae,
    prep_5["X_calibration_scaled"]
)

best_contam_5_threshold = np.quantile(
    best_contam_5_calibration_scores,
    1 - best_contam_5_target_fpr
)

print(
    "Chosen target FPR:",
    round(best_contam_5_target_fpr, 5)
)

print(
    "Threshold:",
    best_contam_5_threshold
)

Chosen target FPR: 0.029
Threshold: 0.0005061811361500931


### Development Evaluation

In [87]:
contam_5_dev_results = []

for test_name, X_dev, y_dev in [
    ("0.2%", X_hpo_02_scaled_5, y_hpo_02),
    ("1%", X_hpo_1_scaled_5, y_hpo_1),
    ("5%", X_hpo_5_scaled_5, y_hpo_5)
]:

    scores = reconstruction_error(
        best_contam_5_ae,
        X_dev
    )

    predictions = (
        scores > best_contam_5_threshold
    ).astype(int)

    result = evaluate_predictions(
        "AE - 5% Contamination + Optuna",
        y_dev,
        predictions
    )

    contam_5_dev_results.append({
        "HPO Set": test_name,
        **result
    })

contam_5_dev_results_df = pd.DataFrame(
    contam_5_dev_results
)

contam_5_dev_results_df.round(4)

,HPO Set,Model,Accuracy,Precision,Recall,F1-Score,FPR,FNR
0,0.2%,AE - 5% Contamination + Optuna,0.9696,0.0035,0.050,0.0065,0.0286,0.950
1,1%,AE - 5% Contamination + Optuna,0.9618,0.0035,0.010,0.0052,0.0286,0.990
2,5%,AE - 5% Contamination + Optuna,0.9240,0.0255,0.014,0.0181,0.0281,0.986


In [88]:
print(
    "Mean Development F1:",
    round(
        contam_5_dev_results_df[
            "F1-Score"
        ].mean(),
        4
    )
)

Mean Development F1: 0.0099


### Observation

The rebuilt model confirms that 5% training contamination severely degrades anomaly detection. Mean development F1 is only 0.0099, and the model misses approximately 98.6% of GoldenEye attacks in the 5% development set.

This indicates that the Autoencoder has learned much of the contaminated attack behaviour as normal reconstruction behaviour.

# Final Holdout Evaluation

The optimized models trained under 0.2%, 1%, and 5% contamination are now evaluated on the same previously unseen holdout datasets.

No further hyperparameter or threshold changes are made.

In [89]:
final_sets = {
    "0.2%": (X_final_02, y_final_02),
    "1%": (X_final_1, y_final_1),
    "5%": (X_final_5, y_final_5)
}

In [90]:
final_scaled_02 = {
    name: (
        transform_with_preprocessing(X, prep_02),
        y
    )
    for name, (X, y) in final_sets.items()
}

final_scaled_1 = {
    name: (
        transform_with_preprocessing(X, prep_1),
        y
    )
    for name, (X, y) in final_sets.items()
}

final_scaled_5 = {
    name: (
        transform_with_preprocessing(X, prep_5),
        y
    )
    for name, (X, y) in final_sets.items()
}

## Final Results - 0.2% Training Contamination

In [91]:
final_results_contam_02 = []

for test_name, (X_scaled, y_true) in final_scaled_02.items():

    scores = reconstruction_error(
        best_contam_02_ae,
        X_scaled
    )

    predictions = (
        scores > best_contam_02_threshold
    ).astype(int)

    result = evaluate_predictions(
        "AE - 0.2% Training Contamination",
        y_true,
        predictions
    )

    final_results_contam_02.append({
        "Training Contamination": "0.2%",
        "Test Set": test_name,
        **result
    })

final_results_contam_02_df = pd.DataFrame(
    final_results_contam_02
)

final_results_contam_02_df.round(4)

,Training Contamination,Test Set,Model,Accuracy,Precision,Recall,F1-Score,FPR,FNR
0,0.2%,0.2%,AE - 0.2% Training Contamination,0.9694,0.0337,0.5167,0.0633,0.0297,0.4833
1,0.2%,1%,AE - 0.2% Training Contamination,0.9657,0.1478,0.5100,0.2292,0.0297,0.4900
2,0.2%,5%,AE - 0.2% Training Contamination,0.9489,0.4900,0.5380,0.5129,0.0295,0.4620


## Final Results - 1% Training Contamination

In [92]:
final_results_contam_1 = []

for test_name, (X_scaled, y_true) in final_scaled_1.items():

    scores = reconstruction_error(
        best_contam_1_ae,
        X_scaled
    )

    predictions = (
        scores > best_contam_1_threshold
    ).astype(int)

    result = evaluate_predictions(
        "AE - 1% Training Contamination",
        y_true,
        predictions
    )

    final_results_contam_1.append({
        "Training Contamination": "1%",
        "Test Set": test_name,
        **result
    })

final_results_contam_1_df = pd.DataFrame(
    final_results_contam_1
)

final_results_contam_1_df.round(4)

,Training Contamination,Test Set,Model,Accuracy,Precision,Recall,F1-Score,FPR,FNR
0,1%,0.2%,AE - 1% Training Contamination,0.9800,0.0073,0.0667,0.0131,0.0182,0.9333
1,1%,1%,AE - 1% Training Contamination,0.9728,0.0409,0.0767,0.0533,0.0182,0.9233
2,1%,5%,AE - 1% Training Contamination,0.9367,0.1891,0.0807,0.1131,0.0182,0.9193


In [93]:
final_results_contam_1 = []

for test_name, (X_scaled, y_true) in final_scaled_1.items():

    scores = reconstruction_error(
        best_contam_1_ae,
        X_scaled
    )

    predictions = (
        scores > best_contam_1_threshold
    ).astype(int)

    result = evaluate_predictions(
        "AE - 1% Training Contamination",
        y_true,
        predictions
    )

    final_results_contam_1.append({
        "Training Contamination": "1%",
        "Test Set": test_name,
        **result
    })

final_results_contam_1_df = pd.DataFrame(
    final_results_contam_1
)

final_results_contam_1_df.round(4)

,Training Contamination,Test Set,Model,Accuracy,Precision,Recall,F1-Score,FPR,FNR
0,1%,0.2%,AE - 1% Training Contamination,0.9800,0.0073,0.0667,0.0131,0.0182,0.9333
1,1%,1%,AE - 1% Training Contamination,0.9728,0.0409,0.0767,0.0533,0.0182,0.9233
2,1%,5%,AE - 1% Training Contamination,0.9367,0.1891,0.0807,0.1131,0.0182,0.9193


## Final Results - 5% Training Contamination

In [94]:
final_results_contam_5 = []

for test_name, (X_scaled, y_true) in final_scaled_5.items():

    scores = reconstruction_error(
        best_contam_5_ae,
        X_scaled
    )

    predictions = (
        scores > best_contam_5_threshold
    ).astype(int)

    result = evaluate_predictions(
        "AE - 5% Training Contamination",
        y_true,
        predictions
    )

    final_results_contam_5.append({
        "Training Contamination": "5%",
        "Test Set": test_name,
        **result
    })

final_results_contam_5_df = pd.DataFrame(
    final_results_contam_5
)

final_results_contam_5_df.round(4)

,Training Contamination,Test Set,Model,Accuracy,Precision,Recall,F1-Score,FPR,FNR
0,5%,0.2%,AE - 5% Training Contamination,0.9703,0.0000,0.0000,0.0000,0.0278,1.0000
1,5%,1%,AE - 5% Training Contamination,0.9626,0.0024,0.0067,0.0035,0.0278,0.9933
2,5%,5%,AE - 5% Training Contamination,0.9244,0.0235,0.0127,0.0165,0.0276,0.9873


## Final Contamination Comparison

In [95]:
final_contamination_results = pd.concat(
    [
        final_results_contam_02_df,
        final_results_contam_1_df,
        final_results_contam_5_df
    ],
    ignore_index=True
)

final_contamination_results.round(4)

,Training Contamination,Test Set,Model,Accuracy,Precision,Recall,F1-Score,FPR,FNR
0,0.2%,0.2%,AE - 0.2% Training Contamination,0.9694,0.0337,0.5167,0.0633,0.0297,0.4833
1,0.2%,1%,AE - 0.2% Training Contamination,0.9657,0.1478,0.5100,0.2292,0.0297,0.4900
2,0.2%,5%,AE - 0.2% Training Contamination,0.9489,0.4900,0.5380,0.5129,0.0295,0.4620
3,1%,0.2%,AE - 1% Training Contamination,0.9800,0.0073,0.0667,0.0131,0.0182,0.9333
4,1%,1%,AE - 1% Training Contamination,0.9728,0.0409,0.0767,0.0533,0.0182,0.9233
5,1%,5%,AE - 1% Training Contamination,0.9367,0.1891,0.0807,0.1131,0.0182,0.9193
6,5%,0.2%,AE - 5% Training Contamination,0.9703,0.0000,0.0000,0.0000,0.0278,1.0000
7,5%,1%,AE - 5% Training Contamination,0.9626,0.0024,0.0067,0.0035,0.0278,0.9933
8,5%,5%,AE - 5% Training Contamination,0.9244,0.0235,0.0127,0.0165,0.0276,0.9873


In [96]:
final_contamination_5 = (
    final_contamination_results[
        final_contamination_results["Test Set"] == "5%"
    ]
    [
        [
            "Training Contamination",
            "Precision",
            "Recall",
            "F1-Score",
            "FPR",
            "FNR"
        ]
    ]
    .sort_values(
        "F1-Score",
        ascending=False
    )
    .reset_index(drop=True)
)

final_contamination_5.round(4)

,Training Contamination,Precision,Recall,F1-Score,FPR,FNR
0,0.2%,0.4900,0.5380,0.5129,0.0295,0.4620
1,1%,0.1891,0.0807,0.1131,0.0182,0.9193
2,5%,0.0235,0.0127,0.0165,0.0276,0.9873


In [97]:
final_contamination_summary = (
    final_contamination_results
    .groupby("Training Contamination")
    ["F1-Score"]
    .mean()
    .reset_index(
        name="Mean Final F1"
    )
    .sort_values(
        "Mean Final F1",
        ascending=False
    )
)

final_contamination_summary.round(4)

,Training Contamination,Mean Final F1
0,0.2%,0.2684
1,1%,0.0598
2,5%,0.0067


# Clean vs Contaminated Training Comparison

In [98]:
clean_vs_contaminated = pd.DataFrame([
    {
        "Training Condition": "Clean Benign",
        "Mean Final F1": 0.5952,
        "5% Precision": 0.8171,
        "5% Recall": 0.9980,
        "5% F1": 0.8986
    },
    {
        "Training Condition": "0.2% Contamination",
        "Mean Final F1": final_contamination_summary.loc[
            final_contamination_summary[
                "Training Contamination"
            ] == "0.2%",
            "Mean Final F1"
        ].values[0],
        "5% Precision": final_contamination_5.loc[
            final_contamination_5[
                "Training Contamination"
            ] == "0.2%",
            "Precision"
        ].values[0],
        "5% Recall": final_contamination_5.loc[
            final_contamination_5[
                "Training Contamination"
            ] == "0.2%",
            "Recall"
        ].values[0],
        "5% F1": final_contamination_5.loc[
            final_contamination_5[
                "Training Contamination"
            ] == "0.2%",
            "F1-Score"
        ].values[0]
    },
    {
        "Training Condition": "1% Contamination",
        "Mean Final F1": final_contamination_summary.loc[
            final_contamination_summary[
                "Training Contamination"
            ] == "1%",
            "Mean Final F1"
        ].values[0],
        "5% Precision": final_contamination_5.loc[
            final_contamination_5[
                "Training Contamination"
            ] == "1%",
            "Precision"
        ].values[0],
        "5% Recall": final_contamination_5.loc[
            final_contamination_5[
                "Training Contamination"
            ] == "1%",
            "Recall"
        ].values[0],
        "5% F1": final_contamination_5.loc[
            final_contamination_5[
                "Training Contamination"
            ] == "1%",
            "F1-Score"
        ].values[0]
    },
    {
        "Training Condition": "5% Contamination",
        "Mean Final F1": final_contamination_summary.loc[
            final_contamination_summary[
                "Training Contamination"
            ] == "5%",
            "Mean Final F1"
        ].values[0],
        "5% Precision": final_contamination_5.loc[
            final_contamination_5[
                "Training Contamination"
            ] == "5%",
            "Precision"
        ].values[0],
        "5% Recall": final_contamination_5.loc[
            final_contamination_5[
                "Training Contamination"
            ] == "5%",
            "Recall"
        ].values[0],
        "5% F1": final_contamination_5.loc[
            final_contamination_5[
                "Training Contamination"
            ] == "5%",
            "F1-Score"
        ].values[0]
    }
])

clean_vs_contaminated.round(4)

,Training Condition,Mean Final F1,5% Precision,5% Recall,5% F1
0,Clean Benign,0.5952,0.8171,0.9980,0.8986
1,0.2% Contamination,0.2684,0.4900,0.5380,0.5129
2,1% Contamination,0.0598,0.1891,0.0807,0.1131
3,5% Contamination,0.0067,0.0235,0.0127,0.0165


### Final Observation

The benign-only Autoencoder provides the strongest and most stable anomaly-detection performance.

As attack contamination increases, the Autoencoder increasingly learns GoldenEye behaviour as part of the reconstruction distribution. This reduces the reconstruction-error separation between normal and attack traffic and causes Recall and F1-score to deteriorate.

Small contamination can be partially compensated for through hyperparameter and threshold optimization, but higher contamination substantially damages anomaly-detection capability.

## Save Results

In [99]:
RESULTS_PATH = Path("../results")
RESULTS_PATH.mkdir(
    parents=True,
    exist_ok=True
)

baseline_contamination_results_df.to_csv(
    RESULTS_PATH / "large_contamination_baseline_results.csv",
    index=False
)

final_contamination_results.to_csv(
    RESULTS_PATH / "large_contamination_final_results.csv",
    index=False
)

final_contamination_summary.to_csv(
    RESULTS_PATH / "large_contamination_summary.csv",
    index=False
)

clean_vs_contaminated.to_csv(
    RESULTS_PATH / "large_clean_vs_contaminated.csv",
    index=False
)

print("Results saved successfully.")

Results saved successfully.


# Conclusion

This experiment evaluated the robustness of a large-scale Autoencoder when GoldenEye attack samples contaminate the training data.

Three contaminated training conditions were studied: 0.2%, 1%, and 5% contamination, while keeping the total training size fixed at 300,000 records.

The results show that reconstruction-based anomaly detection is highly sensitive to contaminated training data. Even small contamination causes the Autoencoder to begin reconstructing attack behaviour as part of its learned normal representation.

Hyperparameter optimization can partially recover performance under low contamination, but the benefit decreases as contamination increases. At 5% contamination, anomaly detection becomes extremely weak because attack traffic is increasingly represented as normal during training.

In comparison, the benign-only large-scale Autoencoder remains substantially more effective, achieving a final 5% anomaly F1-score of 0.8986 with Recall of 0.9980.

These results demonstrate that maintaining a clean reference set of normal traffic is critical for reliable reconstruction-based anomaly detection.